In [1]:
import os
import random
from pathlib import Path

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [2]:
AUTHENTIC_DIR_NAME = "Au_jpg"
TAMPERED_DIR_NAME = "Tp_jpg"

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

IMAGE_SIZE = 224  # standard input size for ResNet
def collect_file_paths(data_dir: str):
    """
    Walks the data directory and returns a list of (filepath, label) tuples.
    label = 0 for authentic, label = 1 for tampered.

    Why a flat list of (path, label) instead of loading images here?
    Loading every image into memory upfront doesn't scale and isn't
    necessary — we only need paths; actual image loading happens
    lazily in __getitem__ , one image at a time,
    which keeps memory usage low no matter how large the dataset is.
    """
    data_dir = Path(data_dir)
    authentic_dir = data_dir / AUTHENTIC_DIR_NAME
    tampered_dir = data_dir / TAMPERED_DIR_NAME

    valid_extensions = {".jpg", ".jpeg", ".png", ".tif", ".bmp"}

    samples = []

    for path in authentic_dir.rglob("*"):
        if path.suffix.lower() in valid_extensions:
            samples.append((str(path), 0))  # 0 = authentic

    for path in tampered_dir.rglob("*"):
        if path.suffix.lower() in valid_extensions:
            samples.append((str(path), 1))  # 1 = tampered

    if len(samples) == 0:
        raise FileNotFoundError(
            f"No images found in {authentic_dir} or {tampered_dir}. "
            "Check that the dataset was downloaded and extracted correctly, "
            "and that AUTHENTIC_DIR_NAME / TAMPERED_DIR_NAME match your "
            "folder names."
        )

    return samples

In [3]:
def split_samples(samples, val_frac=0.15, test_frac=0.15, seed=42):
    random.seed(seed)
    samples = samples.copy()
    random.shuffle(samples)

    n = len(samples)
    n_val = int(n * val_frac)
    n_test = int(n * test_frac)

    test_samples = samples[:n_test]
    val_samples = samples[n_test:n_test + n_val]
    train_samples = samples[n_test + n_val:]

    return train_samples, val_samples, test_samples


In [4]:
def get_transforms(train: bool):
    if train:
        return transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=5),
            transforms.ColorJitter(brightness=0.1, contrast=0.1),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])
    else:
        return transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])

In [5]:
class ForgeryDataset(Dataset):
    """
    A PyTorch Dataset wrapping a list of (filepath, label) samples.

    PyTorch's Dataset class requires two methods:
      - __len__: how many samples total
      - __getitem__: given an index, return (image_tensor, label)

    The DataLoader (built on top of this) handles batching, shuffling, and
    parallel loading (via num_workers) for us.
    """

    def __init__(self, samples, transform):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        # Convert to RGB explicitly: some images in these datasets are
        # grayscale or CMYK, which would break a model expecting 3 channels.
        image = Image.open(path).convert("RGB")
        image = self.transform(image)

        return image, label


In [6]:
def build_dataloaders(data_dir, batch_size=32, num_workers=4):
    """
    Convenience function that ties everything above together:
    collects paths -> splits -> wraps in Dataset -> wraps in DataLoader.

    Returns train_loader, val_loader, test_loader.
    """
    samples = collect_file_paths(data_dir)
    train_samples, val_samples, test_samples = split_samples(samples)

    print(f"Total samples: {len(samples)}")
    print(f"  Train: {len(train_samples)} | Val: {len(val_samples)} | Test: {len(test_samples)}")

    train_ds = ForgeryDataset(train_samples, get_transforms(train=True))
    val_ds = ForgeryDataset(val_samples, get_transforms(train=False))
    test_ds = ForgeryDataset(test_samples, get_transforms(train=False))

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)

    return train_loader, val_loader, test_loader


In [7]:
import torch.nn as nn
from torchvision import models

In [8]:
def build_model(freeze_backbone=True):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

    # model.fc is the final fully-connected (classification) layer.
    # Its default output size is 1000 (for ImageNet's 1000 classes).
    # We replace it with a new Linear layer of our own, output size 2
    # (authentic vs. tampered). Newly created layers have requires_grad=True
    # by default, so this new layer will always be trained even if the rest
    # of the backbone is frozen.
    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, 2)

    return model


In [9]:
def unfreeze_last_n_layers(model, n_layers=2):
    """
    Unfreezes the last n "layer groups" of the ResNet backbone (e.g.
    layer3, layer4) so they can be fine-tuned too, in addition to the final
    classification layer.

    This is typically done as a *second* training phase, after the model
    has already learned a reasonable classifier head with the backbone
    frozen. Unfreezing everything from the start on a small dataset risks
    overfitting or "catastrophic forgetting" of the useful pretrained
    features.
    """
    # ResNet's layers are named layer1, layer2, layer3, layer4 (in order
    # of increasing depth / task-specificity). We unfreeze from the end.
    layer_names = ["layer4", "layer3", "layer2", "layer1"][:n_layers]

    for name in layer_names:
        layer = getattr(model, name)
        for param in layer.parameters():
            param.requires_grad = True

    return model


In [10]:
import argparse
import os

import torch
import torch.nn as nn
from tqdm import tqdm


def parse_args(args=None):
    parser = argparse.ArgumentParser(description="Train a forgery detection classifier")
    parser.add_argument("--data_dir", type=str, required=True)
    parser.add_argument("--epochs", type=int, default=10)
    parser.add_argument("--batch_size", type=int, default=32)
    parser.add_argument("--lr", type=float, default=1e-4)
    parser.add_argument("--unfreeze_epoch", type=int, default=5)
    parser.add_argument("--checkpoint_dir", type=str, default="checkpoints")

    return parser.parse_args(args)

In [11]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    """
    Runs one full pass over the training data.

    model.train() puts the model in training mode — this matters because
    some layers (like BatchNorm) behave differently during training vs.
    evaluation. Forgetting this is a very common bug.
    """
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(loader, desc="Training", leave=False):
        images, labels = images.to(device), labels.to(device)

        # Zero out gradients from the previous batch — PyTorch accumulates
        # gradients by default, so we must clear them each step.
        optimizer.zero_grad()

        outputs = model(images)               # forward pass: raw logits, shape (batch_size, 2)
        loss = criterion(outputs, labels)      # compare predictions to true labels

        loss.backward()                         # backward pass: compute gradients
        optimizer.step()                          # update model weights using those gradients

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)              # predicted class = index of highest logit
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


In [12]:
@torch.no_grad()  # disables gradient tracking -> faster, less memory, since we're not training here
def evaluate(model, loader, criterion, device):
    """
    Runs one full pass over the validation (or test) data, WITHOUT updating
    weights. model.eval() disables dropout and freezes BatchNorm statistics.
    """
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(loader, desc="Evaluating", leave=False):
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


In [7]:
import argparse
import os
import random
from pathlib import Path

import torch
import torch.nn as nn
from tqdm import tqdm
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision import models

# Constants
AUTHENTIC_DIR_NAME = "Au_jpg"
TAMPERED_DIR_NAME = "Tp_jpg"

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

IMAGE_SIZE = 224  # standard input size for ResNet

# Data utility functions and classes
def collect_file_paths(data_dir: str):
    """
    Walks the data directory and returns a list of (filepath, label) tuples.
    label = 0 for authentic, label = 1 for tampered.

    Why a flat list of (path, label) instead of loading images here?
    Loading every image into memory upfront doesn't scale and isn't
    necessary — we only need paths; actual image loading happens
    lazily in __getitem__ , one image at a time,
    which keeps memory usage low no matter how large the dataset is.
    """
    data_dir = Path(data_dir)
    authentic_dir = data_dir / AUTHENTIC_DIR_NAME
    tampered_dir = data_dir / TAMPERED_DIR_NAME

    valid_extensions = {".jpg", ".jpeg", ".png", ".tif", ".bmp"}

    samples = []

    for path in authentic_dir.rglob("*"):
        if path.suffix.lower() in valid_extensions:
            samples.append((str(path), 0))  # 0 = authentic

    for path in tampered_dir.rglob("*"):
        if path.suffix.lower() in valid_extensions:
            samples.append((str(path), 1))  # 1 = tampered

    if len(samples) == 0:
        raise FileNotFoundError(
            f"No images found in {authentic_dir} or {tampered_dir}. "
            "Check that the dataset was downloaded and extracted correctly, "
            "and that AUTHENTIC_DIR_NAME / TAMPERED_DIR_NAME match your "
            "folder names."
        )

    return samples

def split_samples(samples, val_frac=0.15, test_frac=0.15, seed=42):
    random.seed(seed)
    samples = samples.copy()
    random.shuffle(samples)

    n = len(samples)
    n_val = int(n * val_frac)
    n_test = int(n * test_frac)

    test_samples = samples[:n_test]
    val_samples = samples[n_test:n_test + n_val]
    train_samples = samples[n_test + n_val:]

    return train_samples, val_samples, test_samples

def get_transforms(train: bool):
    if train:
        return transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=5),
            transforms.ColorJitter(brightness=0.1, contrast=0.1),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])
    else:
        return transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])

class ForgeryDataset(Dataset):
    """
    A PyTorch Dataset wrapping a list of (filepath, label) samples.

    PyTorch's Dataset class requires two methods:
      - __len__: how many samples total
      - __getitem__: given an index, return (image_tensor, label)

    The DataLoader (built on top of this) handles batching, shuffling, and
    parallel loading (via num_workers) for us.
    """

    def __init__(self, samples, transform):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        # Convert to RGB explicitly: some images in these datasets are
        # grayscale or CMYK, which would break a model expecting 3 channels.
        image = Image.open(path).convert("RGB")
        image = self.transform(image)

        return image, label

def build_dataloaders(data_dir, batch_size=32, num_workers=4):
    """
    Convenience function that ties everything above together:
    collects paths -> splits -> wraps in Dataset -> wraps in DataLoader.

    Returns train_loader, val_loader, test_loader.
    """
    samples = collect_file_paths(data_dir)
    train_samples, val_samples, test_samples = split_samples(samples)

    print(f"Total samples: {len(samples)}")
    print(f"  Train: {len(train_samples)} | Val: {len(val_samples)} | Test: {len(test_samples)}")

    train_ds = ForgeryDataset(train_samples, get_transforms(train=True))
    val_ds = ForgeryDataset(val_samples, get_transforms(train=False))
    test_ds = ForgeryDataset(test_samples, get_transforms(train=False))

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)

    return train_loader, val_loader, test_loader

# Model utility functions
def build_model(freeze_backbone=True):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

    # model.fc is the final fully-connected (classification) layer.
    # Its default output size is 1000 (for ImageNet's 1000 classes).
    # We replace it with a new Linear layer of our own, output size 2
    # (authentic vs. tampered). Newly created layers have requires_grad=True
    # by default, so this new layer will always be trained even if the rest
    # of the backbone is frozen.
    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, 2)

    return model

def unfreeze_last_n_layers(model, n_layers=2):
    """
    Unfreezes the last n "layer groups" of the ResNet backbone (e.g.
    layer3, layer4) so they can be fine-tuned too, in addition to the final
    classification layer.

    This is typically done as a *second* training phase, after the model
    has already learned a reasonable classifier head with the backbone
    frozen. Unfreezing everything from the start on a small dataset risks
    overfitting or "catastrophic forgetting" of the useful pretrained
    features.
    """
    # ResNet's layers are named layer1, layer2, layer3, layer4 (in order
    # of increasing depth / task-specificity). We unfreeze from the end.
    layer_names = ["layer4", "layer3", "layer2", "layer1"][:n_layers]

    for name in layer_names:
        layer = getattr(model, name)
        for param in layer.parameters():
            param.requires_grad = True

    return model

# Training and Evaluation functions
def train_one_epoch(model, loader, optimizer, criterion, device):
    """
    Runs one full pass over the training data.

    model.train() puts the model in training mode — this matters because
    some layers (like BatchNorm) behave differently during training vs.
    evaluation. Forgetting this is a very common bug.
    """
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(loader, desc="Training", leave=False):
        images, labels = images.to(device), labels.to(device)

        # Zero out gradients from the previous batch — PyTorch accumulates
        # gradients by default, so we must clear them each step.
        optimizer.zero_grad()

        outputs = model(images)               # forward pass: raw logits, shape (batch_size, 2)
        loss = criterion(outputs, labels)      # compare predictions to true labels

        loss.backward()                         # backward pass: compute gradients
        optimizer.step()                          # update model weights using those gradients

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)              # predicted class = index of highest logit
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

@torch.no_grad()  # disables gradient tracking -> faster, less memory, since we're not training here
def evaluate(model, loader, criterion, device):
    """
    Runs one full pass over the validation (or test) data, WITHOUT updating
    weights. model.eval() disables dropout and freezes BatchNorm statistics.
    """
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(loader, desc="Evaluating", leave=False):
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

# Argument parsing
def parse_args(args=None):
    parser = argparse.ArgumentParser(description="Train a forgery detection classifier")
    parser.add_argument("--data_dir", type=str, required=True)
    parser.add_argument("--epochs", type=int, default=10)
    parser.add_argument("--batch_size", type=int, default=32)
    parser.add_argument("--lr", type=float, default=1e-4)
    parser.add_argument("--unfreeze_epoch", type=int, default=5)
    parser.add_argument("--checkpoint_dir", type=str, default="checkpoints")

    return parser.parse_args(args)

def main():
    # Simulate command-line arguments for Colab environment.
    # IMPORTANT: You need to replace '/content/your_data_directory' with the actual path
    # to your dataset. For example, if you downloaded and extracted the CASIA 2.0 dataset
    # to '/content/CASIA2', you would use: ['--data_dir', '/content/CASIA2']
    args = parse_args(['--data_dir', '/content/data'])

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    os.makedirs(args.checkpoint_dir, exist_ok=True)

    # --- Data ---
    train_loader, val_loader, _ = build_dataloaders(args.data_dir, batch_size=args.batch_size)

    # --- Model ---
    # Start with the backbone frozen (see model.py docstring for why).
    model = build_model(freeze_backbone=True).to(device)

    # CrossEntropyLoss is the standard choice for multi-class (here,
    # 2-class) classification with raw logit outputs.
    criterion = nn.CrossEntropyLoss()

    # Only pass parameters that require gradients to the optimizer — this
    # matters because the frozen backbone parameters have requires_grad=False,
    # so we don't want the optimizer wasting time/memory tracking them.
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=args.lr,
    )

    best_val_acc = 0.0

    for epoch in range(1, args.epochs + 1):
        print(f"\nEpoch {epoch}/{args.epochs}")

        # --- Progressive unfreezing ---
        # At a chosen epoch, unfreeze the last couple of backbone layers so
        # they can also adapt to forgery-specific features. We also rebuild
        # the optimizer at this point so it picks up the newly-trainable
        # parameters (parameters passed to an optimizer at construction time
        # are fixed; adding requires_grad=True afterward doesn't retroactively
        # add them unless we rebuild it).
        if epoch == args.unfreeze_epoch:
            print("Unfreezing last 2 backbone layers for fine-tuning...")
            model = unfreeze_last_n_layers(model, n_layers=2)
            optimizer = torch.optim.Adam(
                filter(lambda p: p.requires_grad, model.parameters()),
                lr=args.lr / 10,  # smaller LR once more of the network is trainable, to avoid destroying pretrained features
            )

        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)

        print(f"Train loss: {train_loss:.4f} | Train acc: {train_acc:.4f}")
        print(f"Val loss:   {val_loss:.4f} | Val acc:   {val_acc:.4f}")

        # Save the checkpoint only when validation accuracy improves — this
        # protects against overfitting in later epochs (where train acc
        # keeps climbing but val acc plateaus or drops).
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            checkpoint_path = os.path.join(args.checkpoint_dir, "best_model.pth")
            torch.save(model.state_dict(), checkpoint_path)
            print(f"New best model saved (val_acc={val_acc:.4f}) -> {checkpoint_path}")

    print(f"\nTraining complete. Best validation accuracy: {best_val_acc:.4f}")


if __name__ == "__main__":
    main()

Using device: cuda
Total samples: 1496
  Train: 1048 | Val: 224 | Test: 224


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()



Epoch 1/10


Training:   0%|          | 0/33 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Train loss: 0.7304 | Train acc: 0.5010
Val loss:   0.6999 | Val acc:   0.5714
New best model saved (val_acc=0.5714) -> checkpoints/best_model.pth

Epoch 2/10


...

Train loss: 0.3145 | Train acc: 0.8998
Val loss:   0.3482 | Val acc:   0.8750
New best model saved (val_acc=0.8750) -> checkpoints/best_model.pth

Epoch 7/10


Train loss: 0.2585 | Train acc: 0.9103
Val loss:   0.3328 | Val acc:   0.8839
New best model saved (val_acc=0.8839) -> checkpoints/best_model.pth

Epoch 8/10


Train loss: 0.2267 | Train acc: 0.9151
Val loss:   0.3282 | Val acc:   0.8839

Epoch 9/10


Train loss: 0.1950 | Train acc: 0.9303
Val loss:   0.3234 | Val acc:   0.8973
New best model saved (val_acc=0.8973) -> checkpoints/best_model.pth

Epoch 10/10


Train loss: 0.1904 | Train acc: 0.9332
Val loss:   0.3274 | Val acc:   0.8839

Training complete. Best validation accuracy: 0.8973


In [4]:
!unzip -q "/content/drive/MyDrive/dataset_jpg" -d/content/data

In [5]:
import torch
print(torch.cuda.is_available())

True


In [13]:
import argparse
import os
import random
from pathlib import Path

import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
)
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

# Constants
AUTHENTIC_DIR_NAME = "Au_jpg"
TAMPERED_DIR_NAME = "Tp_jpg"

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

IMAGE_SIZE = 224  # standard input size for ResNet

# Data utility functions and classes
def collect_file_paths(data_dir: str):
    """
    Walks the data directory and returns a list of (filepath, label) tuples.
    label = 0 for authentic, label = 1 for tampered.
    """
    data_dir = Path(data_dir)
    authentic_dir = data_dir / AUTHENTIC_DIR_NAME
    tampered_dir = data_dir / TAMPERED_DIR_NAME

    valid_extensions = {".jpg", ".jpeg", ".png", ".tif", ".bmp"}

    samples = []

    for path in authentic_dir.rglob("*"):
        if path.suffix.lower() in valid_extensions:
            samples.append((str(path), 0))  # 0 = authentic

    for path in tampered_dir.rglob("*"):
        if path.suffix.lower() in valid_extensions:
            samples.append((str(path), 1))  # 1 = tampered

    if len(samples) == 0:
        raise FileNotFoundError(
            f"No images found in {authentic_dir} or {tampered_dir}. "
            "Check that the dataset was downloaded and extracted correctly, "
            "and that AUTHENTIC_DIR_NAME / TAMPERED_DIR_NAME match your "
            "folder names."
        )

    return samples

def split_samples(samples, val_frac=0.15, test_frac=0.15, seed=42):
    random.seed(seed)
    samples = samples.copy()
    random.shuffle(samples)

    n = len(samples)
    n_val = int(n * val_frac)
    n_test = int(n * test_frac)

    test_samples = samples[:n_test]
    val_samples = samples[n_test:n_test + n_val]
    train_samples = samples[n_test + n_val:]

    return train_samples, val_samples, test_samples

def get_transforms(train: bool):
    if train:
        return transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=5),
            transforms.ColorJitter(brightness=0.1, contrast=0.1),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])
    else:
        return transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])

class ForgeryDataset(Dataset):
    """
    A PyTorch Dataset wrapping a list of (filepath, label) samples.
    """

    def __init__(self, samples, transform):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")
        image = self.transform(image)
        return image, label

def build_dataloaders(data_dir, batch_size=32, num_workers=4):
    """
    Convenience function that ties everything above together:
    collects paths -> splits -> wraps in Dataset -> wraps in DataLoader.
    """
    samples = collect_file_paths(data_dir)
    train_samples, val_samples, test_samples = split_samples(samples)

    print(f"Total samples: {len(samples)}")
    print(f"  Train: {len(train_samples)} | Val: {len(val_samples)} | Test: {len(test_samples)}")

    train_ds = ForgeryDataset(train_samples, get_transforms(train=True))
    val_ds = ForgeryDataset(val_samples, get_transforms(train=False))
    test_ds = ForgeryDataset(test_samples, get_transforms(train=False))

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers)

    return train_loader, val_loader, test_loader

# Model utility functions
def build_model(freeze_backbone=True):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, 2)

    return model

def unfreeze_last_n_layers(model, n_layers=2):
    layer_names = ["layer4", "layer3", "layer2", "layer1"][:n_layers]
    for name in layer_names:
        layer = getattr(model, name)
        for param in layer.parameters():
            param.requires_grad = True
    return model

def parse_args(args=None):
    parser = argparse.ArgumentParser(description="Evaluate the forgery detection classifier")
    parser.add_argument("--checkpoint", type=str, required=True)
    parser.add_argument("--data_dir", type=str, required=True)
    parser.add_argument("--batch_size", type=int, default=32)
    parser.add_argument("--num_error_examples", type=int, default=12,
                         help="How many misclassified images to save for inspection")
    parser.add_argument("--output_dir", type=str, default="results")
    return parser.parse_args(args)


def denormalize(tensor):
    """
    Reverses the ImageNet normalization applied in dataset.py, so we can
    save misclassified images back to disk as viewable pictures rather than
    normalized tensors (which would look like nonsense if saved directly).
    """
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return tensor * std + mean


@torch.no_grad()
def run_evaluation(model, loader, device):
    """
    Runs the model over the full test set and collects predictions,
    true labels, and the images themselves (needed later to save
    misclassified examples).

    Returns three lists: all_labels, all_preds, all_images (as tensors, so
    we can later pick out and save the misclassified ones).
    """
    model.eval()

    all_labels = []
    all_preds = []
    all_images = []

    for images, labels in loader:
        images_device = images.to(device)
        outputs = model(images_device)
        preds = outputs.argmax(dim=1).cpu()

        all_labels.extend(labels.tolist())
        all_preds.extend(preds.tolist())
        all_images.extend(list(images))  # keep original (CPU, normalized) tensors

    return all_labels, all_preds, all_images


def save_misclassified_examples(labels, preds, images, output_dir, num_examples):
    """
    Saves a handful of misclassified images to disk with filenames
    indicating true vs. predicted label, so you (a human) can look through
    them and reason about *why* the model got them wrong. This qualitative
    error analysis is what turns "here's an accuracy number" into "here's
    what I learned about the model's failure modes" — the latter is what
    the JD is asking for when it mentions "investigating model failures."
    """
    os.makedirs(output_dir, exist_ok=True)

    class_names = {0: "authentic", 1: "tampered"}
    saved = 0

    for i, (label, pred) in enumerate(zip(labels, preds)):
        if label != pred and saved < num_examples:
            img_tensor = denormalize(images[i]).clamp(0, 1)
            img = Image.fromarray(
                (img_tensor.permute(1, 2, 0).numpy() * 255).astype("uint8")
            )
            filename = f"true-{class_names[label]}_pred-{class_names[pred]}_{i}.png"
            img.save(os.path.join(output_dir, filename))
            saved += 1

    print(f"Saved {saved} misclassified examples to {output_dir}/")


def main():
    # Simulate command-line arguments for Colab environment.
    # IMPORTANT: You need to replace 'checkpoints/best_model.pth' with the actual path
    # to your trained model checkpoint and '/content/your_data_directory' with the actual path
    # to your dataset.
    args = parse_args(args=['--checkpoint', 'checkpoints/best_model.pth', '--data_dir', '/content/data'])

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Rebuild the model architecture, then load the trained weights.
    # Note: freeze_backbone doesn't matter here since we're only doing
    # inference (no gradient updates happen during evaluation).
    model = build_model(freeze_backbone=True).to(device)
    model.load_state_dict(torch.load(args.checkpoint, map_location=device))

    _, _, test_loader = build_dataloaders(args.data_dir, batch_size=args.batch_size)

    labels, preds, images = run_evaluation(model, test_loader, device)

    # --- Metrics ---
    acc = accuracy_score(labels, preds)
    precision, recall, f1, support = precision_recall_fscore_support(
        labels, preds, average=None, labels=[0, 1]
    )
    cm = confusion_matrix(labels, preds, labels=[0, 1])

    print("\n=== Test Set Results ===")
    print(f"Accuracy: {acc:.4f}\n")

    print("Per-class metrics:")
    print(f"  Authentic (0): precision={precision[0]:.4f} recall={recall[0]:.4f} f1={f1[0]:.4f} (n={support[0]}) ")
    print(f"  Tampered  (1): precision={precision[1]:.4f} recall={recall[1]:.4f} f1={f1[1]:.4f} (n={support[1]})\n")

    print("Confusion matrix (rows=true, cols=predicted):")
    print("               pred_authentic  pred_tampered")
    print(f"true_authentic      {cm[0][0]:>6}          {cm[0][1]:>6}")
    print(f"true_tampered       {cm[1][0]:>6}          {cm[1][1]:>6}")

    print("\nFull classification report:")
    print(classification_report(labels, preds, target_names=["authentic", "tampered"]))

    # --- Save misclassified examples for qualitative error analysis ---
    error_dir = os.path.join(args.output_dir, "errors")
    save_misclassified_examples(labels, preds, images, error_dir, args.num_error_examples)

if __name__ == "__main__":
    main()

Total samples: 1496
  Train: 1048 | Val: 224 | Test: 224


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()



=== Test Set Results ===
Accuracy: 0.8884

Per-class metrics:
  Authentic (0): precision=0.8647 recall=0.9426 f1=0.9020 (n=122) 
  Tampered  (1): precision=0.9231 recall=0.8235 f1=0.8705 (n=102)

Confusion matrix (rows=true, cols=predicted):
               pred_authentic  pred_tampered
true_authentic         115               7
true_tampered           18              84

Full classification report:
              precision    recall  f1-score   support

   authentic       0.86      0.94      0.90       122
    tampered       0.92      0.82      0.87       102

    accuracy                           0.89       224
   macro avg       0.89      0.88      0.89       224
weighted avg       0.89      0.89      0.89       224

Saved 12 misclassified examples to results/errors/
